In [37]:
import csv

graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4]
}

file_path = '/content/Graph.csv'


with open(file_path, mode='w', newline='') as file:
    writer = csv.writer(file)


    writer.writerow(['Node', 'Neighbors'])


    for node, neighbors in graph.items():

        writer.writerow([node, ', '.join(map(str, neighbors))])

print(f"{file_path} ")


/content/Graph.csv 


1. DFS

In [38]:
def dfs_with_pruning(graph, start_state, goal_state):
    visited = set()

    def dfs(current_state):
        if current_state == goal_state:
            return True
        visited.add(current_state)
        for neighbor in graph.get(current_state, []):
            if neighbor not in visited:
                if dfs(neighbor):
                    return True
        return False

    return dfs(start_state)


2.IDDFS

In [39]:
def iddfs_with_variable_depth(graph, start_state, goal_state, max_depth=100):
    def dfs_with_depth_limit(current_state, depth, limit):
        if depth > limit:
            return False
        if current_state == goal_state:
            return True
        visited.add(current_state)
        for neighbor in graph.get(current_state, []):
            if neighbor not in visited:
                if dfs_with_depth_limit(neighbor, depth + 1, limit):
                    return True
        return False

    visited = set()
    depth = 0
    while depth <= max_depth:
        visited.clear()
        if dfs_with_depth_limit(start_state, 0, depth):
            return True
        depth += 1
    return False


3.UCS

In [40]:
def ucs_with_dynamic_costs(graph, start_state, goal_state, edge_costs):
    frontier = [(0, start_state)]
    came_from = {}
    cost_so_far = {start_state: 0}

    while frontier:
        frontier.sort()
        current_cost, current_state = frontier.pop(0)

        if current_state == goal_state:
            break

        for neighbor in graph.get(current_state, []):
            edge = (str(current_state), str(neighbor))
            if edge not in edge_costs:
                continue
            new_cost = current_cost + edge_costs.get(edge, 1)
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                frontier.append((new_cost, neighbor))
                came_from[neighbor] = current_state


    path = []
    current = goal_state
    while current != start_state:
        path.append(current)
        current = came_from.get(current)
        if current is None:
            return None
    path.append(start_state)
    path.reverse()

    return path


4.Bidirectional BFS

In [41]:
def bidirectional_bfs(graph, start_state, goal_state):
    forward_frontier = [start_state]
    backward_frontier = [goal_state]

    forward_visited = {start_state: None}
    backward_visited = {goal_state: None}

    while forward_frontier and backward_frontier:
        current_state = forward_frontier.pop(0)
        for neighbor in graph.get(current_state, []):
            if neighbor not in forward_visited:
                forward_visited[neighbor] = current_state
                forward_frontier.append(neighbor)
                if neighbor in backward_visited:
                    return reconstruct_path(forward_visited, backward_visited, start_state, goal_state, neighbor)

        current_state = backward_frontier.pop(0)
        for neighbor in graph.get(current_state, []):
            if neighbor not in backward_visited:
                backward_visited[neighbor] = current_state
                backward_frontier.append(neighbor)
                if neighbor in forward_visited:
                    return reconstruct_path(forward_visited, backward_visited, start_state, goal_state, neighbor)

    return None

def reconstruct_path(forward_visited, backward_visited, start_state, goal_state, meeting_point):
    path = []
    current = meeting_point

    while current != start_state:
        path.append(current)
        current = forward_visited[current]
    path.append(start_state)

    current = backward_visited[meeting_point]
    reversed_path = []
    while current != goal_state:
        reversed_path.append(current)
        current = backward_visited[current]
    reversed_path.append(goal_state)

    return path + reversed_path[::-1]


Output

In [42]:
import csv


def load_graph_from_file(file_path):
    graph = {}
    with open(file_path, mode='r') as f:
        reader = csv.reader(f)
        next(reader)

        for row in reader:
            node = int(row[0].strip())
            neighbors = list(map(int, row[1].strip().split(',')))
            graph[node] = neighbors
    return graph


file_path = '/content/Graph.csv'
graph = load_graph_from_file(file_path)


start_state = 0
goal_state = 5


print("DFS with Pruning:", dfs_with_pruning(graph, start_state, goal_state))
print("IDDFS with Variable Depth:", iddfs_with_variable_depth(graph, start_state, goal_state))


edge_costs = {
    ('0', '1'): 1,
    ('1', '0'): 1,
    ('1', '2'): 2,
    ('2', '1'): 2,
    ('1', '3'): 1,
    ('3', '1'): 1,
    ('1', '4'): 2,
    ('4', '1'): 2,
    ('4', '5'): 1,
    ('5', '4'): 1,
    ('2', '5'): 1,
    ('5', '2'): 1
}

print("UCS with Dynamic Costs:", ucs_with_dynamic_costs(graph, start_state, goal_state, edge_costs))
print("Bidirectional BFS:", bidirectional_bfs(graph, start_state, goal_state))



DFS with Pruning: True
IDDFS with Variable Depth: True
UCS with Dynamic Costs: [0, 1, 4, 5]
Bidirectional BFS: [2, 0, 5]
